Question 1: Install Spark and PySpark

In [1]:
# 1) Parar la sesión actual si existe
try:
    spark.stop()
except NameError:
    pass

import os
# 2) Forzar Java 17 en este kernel
#os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-17-openjdk-amd64"
os.environ["JAVA_HOME"] = "C:\\Tools\\jdk-17.0.18+8"
os.environ["PATH"] = f"{os.environ['JAVA_HOME']}\\bin;" + os.environ["PATH"]

# 3) Configurar HADOOP_HOME para Windows (necesario para winutils.exe)
os.environ['HADOOP_HOME'] = 'C:\\hadoop-3.3.1'
# Asegurar que los binarios de hadoop estén en el PATH
os.environ["PATH"] = f"{os.environ['HADOOP_HOME']}\\bin;" + os.environ["PATH"]

# 4) Verifica que ahora ve Java 17
!java -version

openjdk version "17.0.18" 2026-01-20
OpenJDK Runtime Environment Temurin-17.0.18+8 (build 17.0.18+8)
OpenJDK 64-Bit Server VM Temurin-17.0.18+8 (build 17.0.18+8, mixed mode, sharing)


In [2]:
import pyspark
from pyspark.sql import SparkSession

In [3]:
spark = SparkSession.builder \
    .master("local[*]") \
    .appName('test') \
    .getOrCreate()

In [4]:
print(f"Spark version: {spark.version}")

Spark version: 4.1.1


Question 2: Yellow November 2025

In [5]:
!wget https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2025-11.parquet

--2026-03-08 03:04:28--  https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2025-11.parquet
Resolving d37ci6vzurychx.cloudfront.net (d37ci6vzurychx.cloudfront.net)... 3.163.140.145, 3.163.140.127, 3.163.140.18, ...
Connecting to d37ci6vzurychx.cloudfront.net (d37ci6vzurychx.cloudfront.net)|3.163.140.145|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 71134255 (68M) [binary/octet-stream]
Saving to: 'yellow_tripdata_2025-11.parquet'

     0K .......... .......... .......... .......... ..........  0%  419K 2m46s
    50K .......... .......... .......... .......... ..........  0%  469K 2m37s
   100K .......... .......... .......... .......... ..........  0% 5,24M 1m49s
   150K .......... .......... .......... .......... ..........  0%  502K 1m56s
   200K .......... .......... .......... .......... ..........  0% 9,41M 94s
   250K .......... .......... .......... .......... ..........  0% 3,31M 82s
   300K .......... .......... .......... .......... ...

In [5]:
df = spark.read.parquet("yellow_tripdata_2025-11.parquet")

In [6]:
df = df.repartition(4)

In [8]:
df.write.parquet('yellow/2025/11/')

In [9]:
df.show(10)

+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+------------------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|Airport_fee|cbd_congestion_fee|
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+------------------+
|       2| 2025-11-02 08:11:08|  2025-11-02 08:15:21|              1|         1.24|         1|                 N|         186|    

Question 3: Count records

In [10]:
from pyspark.sql import functions as F

df = spark.read.parquet("yellow/2025/11/")

df.filter(F.to_date("tpep_pickup_datetime") == F.lit("2025-11-15")) \
  .select("tpep_pickup_datetime") \
  .count()

162604

Question 4: Longest trip

In [11]:
df.withColumn(
    "trip_hours",
    (F.unix_timestamp("tpep_dropoff_datetime") - F.unix_timestamp("tpep_pickup_datetime")) / 3600.0
).agg(F.max("trip_hours").alias("max_hours")).show()

+-----------------+
|        max_hours|
+-----------------+
|90.64666666666666|
+-----------------+



Question 5: User Interface
Respsonse: 4040

Question 6: Least frequent pickup location zone

In [12]:
!wget https://d37ci6vzurychx.cloudfront.net/misc/taxi_zone_lookup.csv

--2026-03-08 03:31:21--  https://d37ci6vzurychx.cloudfront.net/misc/taxi_zone_lookup.csv
Resolving d37ci6vzurychx.cloudfront.net (d37ci6vzurychx.cloudfront.net)... 3.163.140.37, 3.163.140.18, 3.163.140.145, ...
Connecting to d37ci6vzurychx.cloudfront.net (d37ci6vzurychx.cloudfront.net)|3.163.140.37|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 12331 (12K) [text/csv]
Saving to: 'taxi_zone_lookup.csv'

     0K .......... ..                                         100% 1,91G=0s

2026-03-08 03:31:21 (1,91 GB/s) - 'taxi_zone_lookup.csv' saved [12331/12331]



In [13]:
lookup_df = (spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .csv("taxi_zone_lookup.csv"))

lookup_df.createOrReplaceTempView("zones")

# Ejemplo rápido para verificar
spark.sql("SELECT * FROM zones LIMIT 5").show()

+----------+-------------+--------------------+------------+
|LocationID|      Borough|                Zone|service_zone|
+----------+-------------+--------------------+------------+
|         1|          EWR|      Newark Airport|         EWR|
|         2|       Queens|         Jamaica Bay|   Boro Zone|
|         3|        Bronx|Allerton/Pelham G...|   Boro Zone|
|         4|    Manhattan|       Alphabet City| Yellow Zone|
|         5|Staten Island|       Arden Heights|   Boro Zone|
+----------+-------------+--------------------+------------+



In [14]:
least_pickup_zone = (
    df.join(lookup_df, df.PULocationID == lookup_df.LocationID, "inner")
      .groupBy("Zone")
      .count()
      .orderBy(F.col("count").asc())
      .limit(10)
)

least_pickup_zone.show(truncate=False)

+---------------------------------------------+-----+
|Zone                                         |count|
+---------------------------------------------+-----+
|Arden Heights                                |1    |
|Eltingville/Annadale/Prince's Bay            |1    |
|Governor's Island/Ellis Island/Liberty Island|1    |
|Port Richmond                                |3    |
|Rikers Island                                |4    |
|Rossville/Woodrow                            |4    |
|Great Kills                                  |4    |
|Green-Wood Cemetery                          |4    |
|Jamaica Bay                                  |5    |
|Westerleigh                                  |12   |
+---------------------------------------------+-----+



In [15]:
spark.stop()